In [ ]:
import os

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
MODEL_DIR = "model"

In [ ]:
df = pd.read_csv("dataset/FA-KES-Dataset.csv", encoding="latin1")
true = pd.read_csv("dataset/True.csv")
fake = pd.read_csv("dataset/Fake.csv")

In [ ]:
df

In [ ]:
true['labels'] = 1
fake['labels'] = 0

In [ ]:
true.head()

In [ ]:
fake.head()

In [ ]:
news = pd.concat([fake, true], axis=0)
news

In [ ]:
df.info()

In [ ]:
news.info()

In [ ]:
news = news.drop(['title','subject','date'],axis =1)
news

In [ ]:
news.rename(columns={
    'text':'article_content'
},inplace='True')

In [ ]:
news

In [ ]:
df.isnull().sum()

In [ ]:
df=df.drop(['unit_id','article_title','source','date','location'],axis=1)

In [ ]:
df = pd.concat([df, news],axis = 0)

In [ ]:
df

In [ ]:
df=df.sample(frac=1)

In [ ]:
df.head()

In [ ]:
df.reset_index(inplace=True)
df=df.drop(['index'], axis =1)
df.head()

In [ ]:
def wordopt(text):
    #convert to lowercase
    text=text.lower()

    #remove URLS
    text = re.sub(r'https?://\S+|www\.\S+','', text)

    #remove HTML tags
    text = re.sub(r'<.*?>','',text)

    #remove punctuation
    text = re.sub(r'[^\w\s]','',text)

    #remove digits
    text = re.sub(r'\d','',text)

    #replace newline characters with space
    text = re.sub(r'\n',' ',text)

    return text

In [ ]:
df['article_content'] = df['article_content'].apply(wordopt)

In [ ]:
df['article_content']

In [ ]:
x = df['article_content']
y = df['labels']

In [ ]:
x

In [ ]:
y

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
vectorization = TfidfVectorizer()

In [ ]:
xv_train = vectorization.fit_transform(x_train)

In [ ]:
xv_test = vectorization.transform(x_test)

In [ ]:
xv_train
xv_test

### Logistic Regression

In [ ]:
LR = LogisticRegression()

In [ ]:
LR.fit(xv_train, y_train)

In [ ]:
pred_lr = LR.predict(xv_test)

In [ ]:
LR.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_lr))

### Decision Tree

In [ ]:
DTC = DecisionTreeClassifier()
DTC.fit(xv_train, y_train)

In [ ]:
pred_dtc=DTC.predict(xv_test)

In [ ]:
DTC.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_dtc))

### Random Forest

In [ ]:
rfc = RandomForestClassifier()

In [ ]:
rfc.fit(xv_train, y_train)

In [ ]:
predict_rfc = rfc.predict(xv_test)

In [ ]:
rfc.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_dtc))

### Gradient Boosting

In [ ]:
gbc= GradientBoostingClassifier()

In [ ]:
gbc.fit(xv_train, y_train)

In [ ]:
pred_gbc=gbc.predict(xv_test)

In [ ]:
gbc.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_dtc))

### Testing

In [ ]:
def output_label(n):
    if n == 0:
        return "Fake News!"
    elif n==1:
        return "True News!"

In [ ]:
def manual_testing(news):
    testing_news = {"text": [news]}  # Corrected syntax for defining dictionary
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)  # Assuming 'vectorization' is your vectorizer object

    pred_lr = LR.predict(new_xv_test)
    pred_dtc = DTC.predict(new_xv_test)
    pred_gbc = gbc.predict(new_xv_test)
    pred_rfc = rfc.predict(new_xv_test)

    return "LR Prediction: {} \nDecision Tree Prediction: {} \nGBC Prediction: {} \nRFC Prediction: {}".format(
        output_label(pred_lr[0]),
        output_label(pred_dtc[0]),
        output_label(pred_gbc[0]),
        output_label(pred_rfc[0])
    )

In [ ]:
news_article = str(input())

In [ ]:
manual_testing(news_article)

In [ ]:
joblib.dump(vectorization, os.path.join(MODEL_DIR, "vectorizer.jb"))
joblib.dump(LR, os.path.join(MODEL_DIR, "lr_model.jb"))
joblib.dump(DTC, os.path.join(MODEL_DIR, "dtc_model.jb"))
joblib.dump(gbc, os.path.join(MODEL_DIR, "gbc_model.jb"))
joblib.dump(rfc, os.path.join(MODEL_DIR, "rfc_model.jb"))

In [ ]:

models = {
    "Logistic Regression": LR,
    "Decision Tree": DTC,
    "Random Forest": rfc,
    "Gradient Boosting": gbc, 
}

for name, model in models.items():
    y_pred = model.predict(xv_test)

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=["Fake", "Real"],
        cmap="Blues"
    )
    plt.title(name)
    plt.show()